In [ ]:
import torch
import numpy as np
import os
import time
from torch.utils.data import DataLoader, TensorDataset
from google.colab import drive

# 1. SETUP
drive.mount('/content/drive', force_remount=True)
base_path = "/content/drive/MyDrive/MLA_Final_Project/Datasets_Final/"

# 2. LOAD AUTOENCODER MODEL (From Phase 3)
class DeepAutoencoder(torch.nn.Module):
    def __init__(self, input_dim):
        super(DeepAutoencoder, self).__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 8)
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(8, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(input_dim=25).to(device)
model.load_state_dict(torch.load(os.path.join(base_path, "autoencoder_full.pth")))
model.eval()

# 3. LOAD TEST DATA (Specifically Benign samples for baseline)
X_test = np.load(os.path.join(base_path, "X_test.npy"))
y_test = np.load(os.path.join(base_path, "y_test.npy"))

# We calculate threshold based ONLY on known Normal samples in the test set
normal_samples = torch.Tensor(X_test[y_test == 0]).to(device)

# 4. CALCULATE RECONSTRUCTION ERRORS
print("⚖️ Calculating Baseline Reconstruction Errors...")
criterion = torch.nn.MSELoss(reduction='none') # We need error per sample, not mean

with torch.no_grad():
    _, decoded = model(normal_samples)
    # Calculate MSE per sample: mean across the 25 features
    errors = criterion(decoded, normal_samples).mean(dim=1).cpu().numpy()

# 5. DEFINE THRESHOLD (Mean + 3 Standard Deviations)
# This is a standard statistical approach to define anomalies
mean_error = np.mean(errors)
std_error = np.std(errors)
threshold = mean_error + (3 * std_error)

# Save the threshold value for Phase 5 & 6
np.save(os.path.join(base_path, "anomaly_threshold.npy"), threshold)

# --- 🚀 PROFESSOR-LEVEL INSIGHT: ANOMALY BOUNDARY REPORT ---
print("\n" + "="*50)
print("📊 PHASE 4: THRESHOLD CALCULATION REPORT")
print("="*50)
print(f"✅ Normal Mean Error: {mean_error:.6f}")
print(f"🔍 Standard Deviation: {std_error:.6f}")
print(f"🚨 CALCULATED THRESHOLD: {threshold:.6f}")
print("\n📝 JUSTIFICATION: This threshold serves as the decision")
print("boundary. Any network flow with a reconstruction error")
print(f"above {threshold:.4f} indicates a statistically significant")
print("deviation from the Monday baseline, triggering an IDS alert.")
print("="*50)

Mounted at /content/drive
⚖️ Calculating Baseline Reconstruction Errors...

📊 PHASE 4: THRESHOLD CALCULATION REPORT
✅ Normal Mean Error: 0.027079
🔍 Standard Deviation: 0.130012
🚨 CALCULATED THRESHOLD: 0.417116

📝 JUSTIFICATION: This threshold serves as the decision
boundary. Any network flow with a reconstruction error
above 0.4171 indicates a statistically significant
deviation from the Monday baseline, triggering an IDS alert.
